# Evaluation Visualization
Aggregates evaluation metrics per model for each date/file cohort in `tests/~final` and plots grouped bar charts.
Cohorts: 2025-11-29 models, 2025-11-30 RAG models, and the 2025-11-30 baseline model.


In [ ]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

# Location of evaluation result CSVs
DATA_DIR = Path('tests/~final')
CSV_FILES = sorted(DATA_DIR.glob('*_responses.csv'))

# Mapping of metric columns to display names, grouped for plotting
METRIC_GROUPS = {
    'Accuracy': [
        ('bedrock_correctness', 'Correctness'),
        ('bedrock_faithfulness', 'Faithfulness'),
        ('bedrock_citation_precision', 'Citation precision'),
        ('bedrock_citation_coverage', 'Citation coverage'),
        ('bedrock_completeness', 'Completeness'),
        ('nvidia_answer_accuracy', 'NVIDIA answer accuracy'),
    ],
    'Actionability': [
        ('bedrock_logical_coherence', 'Coherence'),
        ('bedrock_helpfulness', 'Helpfulness'),
        ('bedrock_refusal', 'Refusal'),
        ('bedrock_relevance', 'Relevance'),
        ('bedrock_harmfulness', 'Harmfulness'),
        ('bedrock_stereotyping', 'Stereotyping'),
    ],
    'Readability/Sentiment': [
        ('flesch_grade', 'Flesch grade'),
        ('flesch_reading_ease_score', 'Flesch readability'),
        ('nltk_sentiment', 'NLTK sentiment'),
    ],
}

# Desired cohort ordering for plotting
FILE_GROUP_ORDER = [
    '2025-11-29 models',
    '2025-11-30 RAG models',
    '2025-11-30 baseline',
]

def normalize_model_name(path: Path) -> str:
    stem = path.stem
    stem = re.sub(r'^\d{4}-\d{2}-\d{2}_', '', stem)
    stem = re.sub(r'_responses$', '', stem)
    return stem


def categorize_file(path: Path) -> str:
    name = path.name
    if '2025-11-29' in name:
        return '2025-11-29 models'
    if '2025-11-30' in name and 'baseline' in name:
        return '2025-11-30 baseline'
    if '2025-11-30' in name:
        return '2025-11-30 RAG models'
    return 'Other'

rows = []
for csv_path in CSV_FILES:
    file_group = categorize_file(csv_path)
    model = normalize_model_name(csv_path)
    df = pd.read_csv(csv_path)
    numeric = df.select_dtypes(include='number')

    for group_name, metrics in METRIC_GROUPS.items():
        for col, label in metrics:
            if col not in numeric.columns:
                continue
            rows.append({
                'file_group': file_group,
                'group': group_name,
                'metric': label,
                'model': model,
                'avg_value': numeric[col].mean(),
            })

agg_df = pd.DataFrame(rows)

# Enforce intended metric ordering within each group for plotting
for group_name, metrics in METRIC_GROUPS.items():
    order = [label for _, label in metrics]
    mask = agg_df['group'] == group_name
    agg_df.loc[mask, 'metric'] = pd.Categorical(agg_df.loc[mask, 'metric'], categories=order, ordered=True)

agg_df.head()


In [ ]:
def plot_group_for_file_group(file_group: str, metric_group: str, figsize=(12, 6)):
    subset = agg_df[(agg_df['file_group'] == file_group) & (agg_df['group'] == metric_group)].copy()
    if subset.empty:
        print(f'No data for {file_group} / {metric_group}')
        return

    order = [label for _, label in METRIC_GROUPS[metric_group]]
    subset['metric'] = pd.Categorical(subset['metric'], categories=order, ordered=True)
    subset = subset.sort_values('metric')

    pivot = subset.pivot(index='metric', columns='model', values='avg_value')
    ax = pivot.plot(kind='bar', figsize=figsize, rot=45)
    ax.set_title(f'{metric_group} metrics by model — {file_group}')
    ax.set_xlabel('Metric')
    ax.set_ylabel('Average value')
    ax.legend(title='Model')
    plt.tight_layout()
    plt.show()

for file_group in FILE_GROUP_ORDER:
    print(f'
=== {file_group} ===')
    plot_group_for_file_group(file_group, 'Accuracy')
    plot_group_for_file_group(file_group, 'Actionability')
    plot_group_for_file_group(file_group, 'Readability/Sentiment', figsize=(10, 5))
